### Importando Bibliotecas Básicas

In [95]:

from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from pathlib import Path
import os
import matplotlib.pyplot as plt
import seaborn as sns
from statannotations.Annotator import Annotator
import pingouin as pg


### Exploraçao

In [96]:
# Criando um caminho universal para que rode em qualquer maquina
pasta_atual = Path.cwd()
pasta_banco = pasta_atual.parent / 'database'

In [97]:
# Carregando os arquivos e modificando os tipos de dados

contas_df = pd.read_csv(pasta_banco/'account.asc', sep=';', dtype= {'account_id':str, 'district_id': str})
cartoes_df = pd.read_csv(pasta_banco/'card.asc', sep=';', dtype= {'card_id': str,'disp_id':str})
clientes_df = pd.read_csv(pasta_banco/'client.asc', sep=';', dtype= {'client_id':str,'district_id':str, 'birth_number': str})
cliente_conta_df = pd.read_csv(pasta_banco/'disp.asc', sep=';', dtype= {'disp_id':str, 'client_id':str,'account_id':str})
emprestimo_df = pd.read_csv(pasta_banco/'loan.asc', sep=';', dtype = {'loan_id':str,'account_id':str})
ordens_df = pd.read_csv(pasta_banco/'order.asc', sep=';', dtype = {'order_id':str,'account_id':str})
distrito_df = pd.read_csv(pasta_banco/'district.asc', sep=';',na_values=['?'] ,dtype = {'A1':str,'A11':float,'A12':float,'A15':'Int64'})
transacoes_df = pd.read_csv(pasta_banco/'trans.asc', sep=';',dtype={'trans_id' : str,'account_id' : str, 'bank':str}) 


### Correção de Inconsistencias e Traduçao de Linhas e Colunas

In [98]:
# tabela contas_df

contas_df = contas_df.rename(columns={
    'account_id': 'id_conta',
    'district_id': 'id_distrito',
    'date': 'data',
    'frequency': 'frequencia'
})

ajustar = {
    'POPLATEK MESICNE': 'emissao_mensal',
    'POPLATEK PO OBRATU': 'emissao_semanal',
    'POPLATEK TYDNE': 'emissao_pos_transacao'
}


contas_df['frequencia'] = contas_df['frequencia'].replace(ajustar)
contas_df.head(2)

#-----------------------------------------------------------------------------

# tabela contas_df

cartoes_df = cartoes_df.rename(columns={
    'card_id': 'id_cartao',
    'disp_id': 'id_disp',
    'type': 'tipo_cartao',
    'issued': 'data'
})


cartoes_df.sample(10)

#-----------------------------------------------------------------------------

# tabela clientes_df

clientes_df = clientes_df.rename(columns={
    'client_id': 'id_cliente',
    'district_id': 'id_distrito',
    'gender': 'genero',
    'birth_date': 'data_nascimento'
})




#clientes_df.sample(10)


#-----------------------------------------------------------------------------

# tabela cliente_conta_df

cliente_conta_df = cliente_conta_df.rename(columns={
    'disp_id	': 'id_disp',
    'client_id': 'id_cliente',
    'account_id': 'id_conta',
    'type': 'tipo_acesso'
})

ajustar = {
    'OWNER': 'proprietario',
    'DISPONENT': 'usuario'
}

cliente_conta_df['tipo_acesso'] = cliente_conta_df['tipo_acesso'].replace(ajustar)
#cliente_conta_df.sample(10)
#-----------------------------------------------------------------------------

# tabela emprestimo_df

emprestimo_df = emprestimo_df.rename(columns={
    'loan_id': 'id_emprestimo',
    'account_id': 'id_conta',
    'date': 'data',
    'amount': 'valor_total',
    'duration': 'prazo_meses',
    'payments': 'valor_parcela',
    'status': 'status_pagamento'
})

ajustar = {
    'A': 'Contrato Encerrado - Quitado',
    'B': 'Contrato Encerrado - Inadimplente',
    'C': 'Contrato Ativo - Regular',
    'D': 'Contrato Ativo - Inadimplente'
}


emprestimo_df['status_pagamento'] = emprestimo_df['status_pagamento'].replace(ajustar)
#emprestimo_df.sample(10)

#-----------------------------------------------------------------------------


# tabela ordens_df

ordens_df = ordens_df.rename(columns={
    'order_id': 'id_ordem',
    'account_id': 'id_conta',
    'bank_to': 'banco_destino',
    'account_to': 'conta_destino',
    'amount': 'valor_debito',
    'k_symbol': 'categoria_pagamento'
})

ajustar = {
    'SIPO': 'Despesas Domesticas',
    'UVER': 'Parcela de Emprestimo',
    'POJISTNE': 'Pagamento de Seguro',
    'LEASING': 'Financiamento de Veículos e Máquinas'
}


ordens_df['categoria_pagamento'] = ordens_df['categoria_pagamento'].replace(ajustar)
#ordens_df.sample(10)


#-----------------------------------------------------------------------------

# tabela distrito_df

distrito_df = distrito_df.rename(columns={
    'A1': 'id_distrito',
    'A2': 'nome_distrito',
    'A3': 'regiao',
    'A4': 'num_habitantes',
    'A5': 'municipios_ate_499_hab',
    'A6': 'municipios_500_1999_hab',
    'A7': 'municipios_2000_9999_hab',
    'A8': 'municipios_mais_10000_hab',
    'A9': 'num_cidades',
    'A10': 'proporcao_populacao_urbana',
    'A11': 'salario_medio',
    'A12': 'taxa_desemprego_95',
    'A13': 'taxa_desemprego_96',
    'A14': 'num_empreendedores_por_1000_hab',
    'A15': 'crimes_cometidos_95',
    'A16': 'crimes_cometidos_96'
})


distrito_df['regiao'] = distrito_df['regiao'].str.capitalize()

#distrito_df.sample(10)

#-----------------------------------------------------------------------------

# tabela transacoes_df

transacoes_df = transacoes_df.rename(columns={
    'trans_id': 'id_transacao',
    'account_id': 'id_conta',
    'date': 'data',
    'type': 'tipo_movimentacao',
    'operation': 'modo_operacao',
    'amount': 'valor_transacao',
    'balance': 'saldo_apos_transacao',
    'k_symbol': 'categoria_transacao',
    'bank': 'banco_parceiro',
    'account': 'conta_parceiro'
})

tipo_mov = {
    'PRIJEM': 'Credito (+)',
    'VYDAJ': 'Debito (-)',
    'VYBER': 'Debito (-)'
}


tipo_ope = {
    'VKLAD': 'Deposito em Dinheiro',
    'PREVOD Z UCTU': 'Transferencia Recebida',
    'VYBER': 'Saque em Dinheiro',
    'PREVOD NA UCET': 'Transferencia Enviada',
    'VYBER KARTOU': 'Saque com Cartao'
}

tipo_trans ={
    'POJISTNE': 'Pagamento de Seguro',
    'SLUZBY': 'Tarifa Bancaria',
    'UROK': 'Juros Recebidos',
    'SANKC. UROK': 'Juros de Multa',
    'SIPO': 'Contas de Consumo',
    'DUCHOD': 'Aposentadoria / Pensao',
    'UVER': 'Parcela de Emprestimo'
}




transacoes_df['tipo_movimentacao'] = transacoes_df['tipo_movimentacao'].replace(tipo_mov)
transacoes_df['modo_operacao'] = transacoes_df['modo_operacao'].replace(tipo_ope)
transacoes_df['categoria_transacao'] = transacoes_df['categoria_transacao'].replace(tipo_trans)
#transacoes_df.sample(2)

In [ ]:
# 2. Definição do Dicionário de Mapeamento Semântico
# Nota de Qualidade de Dados: 
# A tag 'BAD' denota estritamente um padrão deficiente de qualidade e conformidade (inadimplência estrutural).
mapeamento_status = {
    'A': 'GOOD', # Contrato finalizado, sem pendências
    'B': 'BAD',  # Contrato finalizado, não pago (Padrão deficiente)
    'C': 'GOOD', # Contrato em andamento, regular
    'D': 'BAD'   # Contrato em andamento, em dívida (Padrão deficiente)
}

# 3. Aplicação da regra de negócio criando uma nova coluna analítica
# Preservamos a coluna original 'status' caso seja necessária para auditoria, 
# e criamos 'status_qualidade' para uso no Power BI.
emprestimo_df['status_qualidade'] = emprestimo_df['status'].map(mapeamento_status)

# Verificando a distribuição da nova categorização
print(emprestimo_df['status_qualidade'].value_counts())

In [ ]:
# Recolhendo amostras
transacoes_df.sample(5)

In [ ]:
# Sumario dos dados

transacoes_df.info()

In [ ]:
# Percentual de nulos
print('Percentual de nulos por coluna:')
print((ordens_df.isnull().sum() / len(ordens_df) * 100).round(2))

contas_df = pd.read_csv(pasta_banco/'account.asc', sep=';', dtype= {'account_id':str, 'district_id': str})
cartoes_df = pd.read_csv(pasta_banco/'card.asc', sep=';', dtype= {'card_id': str,'disp_id':str})
clientes_df = pd.read_csv(pasta_banco/'client.asc', sep=';', dtype= {'client_id':str,'district_id':str, 'birth_number': str})
cliente_conta_df = pd.read_csv(pasta_banco/'disp.asc', sep=';', dtype= {'disp_id':str, 'client_id':str,'account_id':str})
emprestimo_df = pd.read_csv(pasta_banco/'loan.asc', sep=';', dtype = {'loan_id':str,'account_id':str})
ordens_df = pd.read_csv(pasta_banco/'order.asc', sep=';', dtype = {'order_id':str,'account_id':str})
distrito_df = pd.read_csv(pasta_banco/'district.asc', sep=';',na_values=['?'] ,dtype = {'A1':str,'A11':float,'A12':float,'A15':'Int64'})
transacoes_df = pd.read_csv(pasta_banco/'trans.asc', sep=';',dtype={'trans_id' : str,'account_id' : str, 'bank':str}) 

In [104]:
distrito_df.sample(5)

,id_distrito,nome_distrito,regiao,num_habitantes,municipios_ate_499_hab,municipios_500_1999_hab,municipios_2000_9999_hab,municipios_mais_10000_hab,num_cidades,proporcao_populacao_urbana,salario_medio,taxa_desemprego_95,taxa_desemprego_96,num_empreendedores_por_1000_hab,crimes_cometidos_95,crimes_cometidos_96
46,47,Pardubice,East bohemia,162580,83,26,5,1,6,72.8,9538.0,1.51,1.81,111,6079,5410
54,55,Brno - venkov,South moravia,157042,49,70,18,0,9,33.9,8743.0,1.88,2.43,111,3659,3894
24,25,Klatovy,West bohemia,88757,60,33,3,2,10,61.7,8554.0,2.47,2.68,113,1822,2218
62,63,Vyskov,South moravia,86513,38,36,5,1,5,50.5,8288.0,3.79,4.52,110,1562,1460
2,3,Beroun,Central bohemia,75232,55,26,4,1,5,41.7,8980.0,1.95,2.21,111,2824,2813


In [106]:
# Verificação de dados unicos para identificaçao de possiveis erros
print(distrito_df['regiao'].unique())


<StringArray>
[         'Prague', 'Central bohemia',   'South bohemia',    'West bohemia',
   'North bohemia',    'East bohemia',   'South moravia',   'North moravia']
Length: 8, dtype: str


In [ ]:
# Conta quantas linhas possuem o espaço em branco
quantidade_vazios = (ordens_df['banco_destino'] == ' ').sum()
total_linhas = len(ordens_df)
porcentagem = (quantidade_vazios / total_linhas) * 100

print(f"Linhas vazias: {quantidade_vazios} ({porcentagem:.2f}%)")

In [ ]:
# Substitui o espaço vazio por 'Não Informado'
transacoes_df['categoria_transacao'] = transacoes_df['categoria_transacao'].replace(' ', 'Não Informado')

# Verificação
print(transacoes_df['categoria_transacao'].unique())

In [ ]:
# Substitui o espaço vazio por ('Não Informado')
ordens_df['categoria_pagamento'] = ordens_df['categoria_pagamento'].replace(' ', 'Não Informado')

# Verificação
print(ordens_df['categoria_pagamento'].unique())

In [ ]:
# Estatisticas descritivas dos dados

def descritivas(data):
  variaveis = data.select_dtypes(include=np.number)
  desc = variaveis.describe().T
  desc["CV"] = desc["std"]/desc["mean"]
  desc["Skew"] = variaveis.skew()
  desc["Kurtosis"] = variaveis.kurt()
  ordered_cols = [
      "count", "mean", "std", "CV",
      "min", "25%", "50%", "75%", "max",
      "Skew", "Kurtosis"
  ]
  desc = desc[ordered_cols]
  return desc.round(2)


In [ ]:
# analise descritiva da coluna CL_FHL (quantidade de filhos por cliente)
descritivas(transacoes_df)

##### Coluna "valor_transacao"
- Alta variabilidade: Verificamos que o desvio padrão da coluna está muito alto (cerca de 10.000) em comparação com a sua média (cerca de 6.000). Isso nos mostra que os valores das transações variam drasticamente, o que explica o elevado Coeficiente de Variação de 161%.

- Presença de Outliers: Por mais que a média dos dados seja de 6.000, a nossa mediana (o valor que representa 50% da nossa base) é de apenas 2.100. Essa grande diferença (média muito maior que a mediana) indica claramente que a média está sendo puxada para cima por outliers (transações de valores muito altos).

- Assimetria Positiva: A hipótese acima é confirmada pela métrica de Skewness (Assimetria), que está em cerca de 2.59. Isso comprova que a distribuição dos dados está concentrada em valores menores, com uma cauda longa se estendendo para a direita.

- Valores Extremos: A medida de Kurtosis (Curtose) alta reforça esse cenário, nos explicando que a distribuição possui caudas pesadas. Ou seja, temos uma grande concentração de transações de valor muito baixo (próximas a 0) e, ao mesmo tempo, a presença de transações com valores atipicamente altos.

- Comportamento de Microtransações: Cerca de 1/4 das nossas transações (o 1º Quartil) se concentram em valores de até 140. Do ponto de vista de negócios, isso significa que provavelmente grande parte do nosso volume é composto por tarifas bancárias, pequenos saques, juros ou compras corriqueiras.

##### Coluna "saldo_apos_transacao"
- Uso de Limite/Cheque Especial: Verificamos a presença de saldos negativos, indicando que o banco possui alguma política que permite aos clientes ficarem com a conta negativada. Isso levanta a hipótese de um uso frequente de crédito extra ou cheque especial por parte da base.

- Saúde Financeira e Clientes Alta Renda: A análise permite presumir que boa parte da nossa base de clientes possui uma saúde financeira estável, sem grandes anomalias no dia a dia. Contudo, as métricas de Skew e Kurtosis mostram que a curva de saldo está sendo fortemente puxada para a direita, um indicativo claro de que temos uma parcela de clientes com alto poder aquisitivo (ou grandes empresas) inflando a média geral de saldos.

In [ ]:
transacoes_df.sample(5)

In [ ]:
# Analise Descritiva por segmentaçao
transacoes_df.groupby('operation').apply(descritivas)

### Tabela "transacoes_df
#### Coluna "tipo_movimentacao"

##### Contagem
- Temos cerca de 651 mil transações de Débito contra 405 mil de Crédito, o que é um alto indicativo que uma boa parcela dos nossos clientes mais gastam do que recebem, oque representa que temos algum uso de cheque especial dentro da nossa base

##### Media e Mediana
- Foi identificado que temos uma grande diferença entre a media e a mediana, pelo menos 50% dos nossos clientes, gastam ate 2.200 reais, a media não reflete a mesma realida, pois muito provavelmente, temos outliers puxando esta media para cima

##### Coeficiente de Variação e Desvio Padrao
- Valores acima de 1 indicam altissima dispersao, o que nos leva a considerar que dentro da nossa base de dados existe uma variaçao de tipos de clientes (baixa renda e alta renda) ou (contas PJ misturados com PF)

- **Possivel Ação:** clusterizar os clientes iremos usar **Machine Learning**

##### Min / Max e Quartis
- Foi possivel verificar que temos valores negativos nas duas colunas "valor_transacao" e "saldo_apos_transacao" respectivamente de -31.000 e -41.000, oque diz que nossos clientes estao usando cheque especial ou limite de credito

##### Skew / Kurtosis
- Nossos dados nao Assimetricos (Skew) ou seja, temos uma concentraçao massiva de valores pequenos e uma cauda longa de valores grandes
- Geralmente, para uma distribuiçao normal de Kurtosis, se espera valores de ate 3, mas no caso da coluna "tipo_movimento" Debito para a coluna ""valor_transacao", tivemos um valor de 15.68, oque nos diz que ha anomalias ou comportamentos atipicos dentro da nossa base de dados.
    - Podemos levantar algumas poucas hipoteses, pode ser fraude ou compra de maquinarios por empresas (PJ)
    - **Ação:** Isolar estes dados e investigalos


------------------------------------------------------------------------------------------------------------

#### Coluna "modo_operacao"

##### Contagem
- Podemos identificar que dentro da coluna "modo_operacao" temos cerca de 41% dos nossos clientes dentro da operação **Saque em Dinheiro** e cerca de 19% dentro da operação **Transferencia Enviada**

##### Media e Mediana
- Foi identificado que temos outlier

##### Coeficiente de Variação e Desvio Padrao
- 

##### Min / Max e Quartis
- 

##### Skew / Kurtosis
- 

In [ ]:
# Linhas duplicadas
transacoes_df.duplicated().sum()

### Tratamento de Datas

In [ ]:
# Tratamento de datas na tabela de transacoes_df
# transformamos o número inteiro em texto (string)
transacoes_df['date'] = transacoes_df['date'].astype(str)

# convertendo para datetime usando o formato YYMMDD (padrão do banco Berka, conforme documentacao deste dataset)
transacoes_df['date'] = pd.to_datetime(
    transacoes_df['date'],
    format='%y%m%d',
    errors='coerce'
)


# ____________________________________________________________________________________

# Tratamento de datas na tabela de emprestimo_df
emprestimo_df['date'] = emprestimo_df['date'].astype(str)

emprestimo_df['date'] = pd.to_datetime(
    emprestimo_df['date'],
    format='%y%m%d',
    errors='coerce'
)

# ____________________________________________________________________________________

# Tratamento de datas na tabela de cartoes_df
cartoes_df['issued'] = cartoes_df['issued'].astype(str).str[:6]

cartoes_df['issued'] = pd.to_datetime(
    cartoes_df['issued'],
    format='%y%m%d',
    errors='coerce'
)
# ____________________________________________________________________________________

# Tratamento de datas na tabela de contas_df
contas_df['date'] = contas_df['date'].astype(str).str[:6]
contas_df['date'] = pd.to_datetime(
    contas_df['date'],
    format= '%y%m%d',
    errors='coerce'
)


# verificação rápida para garantir que funcionou e ver o real alcance dos dados
print(f"Data Mínima das Transações: {contas_df['date'].min()}")
print(f"Data Máxima das Transações: {contas_df['date'].max()}")

In [ ]:
# Tratamento de datas na tabela de client_df

clientes_df['birth_number'] = clientes_df['birth_number'].str.zfill(6)

# criando variaveis para que recebam o dia,ano,mes fatiado para facilitar a analise e tratamento
# e alterando o tipo de dados para realizar 

ano = clientes_df['birth_number'].str[:2]
mes = clientes_df['birth_number'].str[2:4].astype(int)
dia = clientes_df['birth_number'].str[4:]

# iremos criar a coluna o genero dos clientes com base no mes

clientes_df['gender'] = np.where(mes > 50, 'F', 'M')

# arrumando o mes para que ele retorne se somente for mulher

mes_limpo = np.where(mes > 50, mes - 50, mes)

# Volta o mês para texto com 2 dígitos (para que o mês 8 volte a ser '08')

mes_limpo_str = pd.Series(mes_limpo).astype(str).str.zfill(2)

# remontando a data e convertendo para datetime e forçando a inserçao do seculo, estipulando que os clientes nasceram no seculo 19

clientes_df['birth_date'] = pd.to_datetime('19' + ano + mes_limpo_str + dia, format='%Y%m%d')

# coluna original birth_date apagada 
clientes_df = clientes_df.drop(columns=['birth_number'])

### Criando dimCalendario

In [ ]:

# identificando o ano inial e final para criar a tabela dim calendario
ano_inicio = transacoes_df['date'].min().year
ano_fim = transacoes_df['date'].max().year

# 2. Definir o intervalo completo (01/Jan do primeiro ano até 31/Dez do último ano)
data_inicio_completa = f'{ano_inicio}-01-01'
data_fim_completa = f'{ano_fim}-12-31'

# 3. Criar a sequência contínua de dias
datas_calendario = pd.date_range(start=data_inicio_completa, end=data_fim_completa)

# 4. Criar o DataFrame da dCalendario
dimCalendario = pd.DataFrame({'date': datas_calendario})

# 5. Extrair as colunas derivadas de forma segura (usando a própria dimCalendario)
dimCalendario['date_year'] = dimCalendario['date'].dt.year
dimCalendario['date_month'] = dimCalendario['date'].dt.month
dimCalendario['date_quarter'] = dimCalendario['date'].dt.quarter
dimCalendario['date_year_month'] = dimCalendario['date'].dt.strftime('%Y-%m')
dimCalendario['date_week'] = dimCalendario['date'].dt.isocalendar().week
dimCalendario['date_day'] = dimCalendario['date'].dt.day
dimCalendario['date_day_name'] = dimCalendario['date'].dt.day_name()

# Visualizar o resultado final do calendário
dimCalendario

### Correção de Inconsistencias e Traduçao de Linhas e Colunas

In [ ]:
# tabela contas_df

contas_df = contas_df.rename(columns={
    'account_id': 'id_conta',
    'district_id': 'id_distrito',
    'date': 'data',
    'frequency': 'frequencia'
})

ajustar = {
    'POPLATEK MESICNE': 'emissao_mensal',
    'POPLATEK PO OBRATU': 'emissao_semanal',
    'POPLATEK TYDNE': 'emissao_pos_transacao'
}


contas_df['frequencia'] = contas_df['frequencia'].replace(ajustar)
contas_df.head(2)

In [ ]:
# tabela contas_df

cartoes_df = cartoes_df.rename(columns={
    'card_id': 'id_cartao',
    'disp_id': 'id_disp',
    'type': 'tipo_cartao',
    'issued': 'data'
})


cartoes_df.sample(10)

In [ ]:
# tabela clientes_df

clientes_df = clientes_df.rename(columns={
    'client_id': 'id_cliente',
    'district_id': 'id_distrito',
    'gender': 'genero',
    'birth_date': 'data_nascimento'
})




clientes_df.sample(10)

In [ ]:
# tabela cliente_conta_df

cliente_conta_df = cliente_conta_df.rename(columns={
    'disp_id	': 'id_disp',
    'client_id': 'id_cliente',
    'account_id': 'id_conta',
    'type': 'tipo_acesso'
})

ajustar = {
    'OWNER': 'proprietario',
    'DISPONENT': 'usuario'
}

cliente_conta_df['tipo_acesso'] = cliente_conta_df['tipo_acesso'].replace(ajustar)
cliente_conta_df.sample(10)


In [ ]:
# tabela emprestimo_df

emprestimo_df = emprestimo_df.rename(columns={
    'loan_id': 'id_emprestimo',
    'account_id': 'id_conta',
    'date': 'data',
    'amount': 'valor_total',
    'duration': 'prazo_meses',
    'payments': 'valor_parcela',
    'status': 'status_pagamento'
})

ajustar = {
    'A': 'quitado_regular',
    'B': 'inadimplente',
    'C': 'em andamento_regular',
    'D': 'em andamento_crédito_defectivo'
}


emprestimo_df['status_pagamento'] = emprestimo_df['status_pagamento'].replace(ajustar)
emprestimo_df.sample(10)

In [ ]:
# tabela ordens_df

ordens_df = ordens_df.rename(columns={
    'order_id': 'id_ordem',
    'account_id': 'id_conta',
    'bank_to': 'banco_destino',
    'account_to': 'conta_destino',
    'amount': 'valor_debito',
    'k_symbol': 'categoria_pagamento'
})

ajustar = {
    'SIPO': 'Despesas Domesticas',
    'UVER': 'Parcela de Emprestimo',
    'POJISTNE': 'Pagamento de Seguro',
    'LEASING': 'Financiamento de Veículos e Máquinas'
}


ordens_df['categoria_pagamento'] = ordens_df['categoria_pagamento'].replace(ajustar)
ordens_df.sample(10)

In [ ]:
# tabela distrito_df

distrito_df = distrito_df.rename(columns={
    'A1': 'id_distrito',
    'A2': 'nome_distrito',
    'A3': 'regiao',
    'A4': 'num_habitantes',
    'A5': 'municipios_ate_499_hab',
    'A6': 'municipios_500_1999_hab',
    'A7': 'municipios_2000_9999_hab',
    'A8': 'municipios_mais_10000_hab',
    'A9': 'num_cidades',
    'A10': 'proporcao_populacao_urbana',
    'A11': 'salario_medio',
    'A12': 'taxa_desemprego_95',
    'A13': 'taxa_desemprego_96',
    'A14': 'num_empreendedores_por_1000_hab',
    'A15': 'crimes_cometidos_95',
    'A16': 'crimes_cometidos_96'
})


distrito_df['regiao'] = distrito_df['regiao'].str.capitalize()

distrito_df.sample(10)

In [ ]:
# tabela ordens_df

transacoes_df = transacoes_df.rename(columns={
    'trans_id': 'id_transacao',
    'account_id': 'id_conta',
    'date': 'data',
    'type': 'tipo_movimentacao',
    'operation': 'modo_operacao',
    'amount': 'valor_transacao',
    'balance': 'saldo_apos_transacao',
    'k_symbol': 'categoria_transacao',
    'bank': 'banco_parceiro',
    'account': 'conta_parceiro'
})

tipo_mov = {
    'PRIJEM': 'Credito (+)',
    'VYDAJ': 'Debito (-)',
    'VYBER': 'Debito (-)'
}


tipo_ope = {
    'VKLAD': 'Deposito em Dinheiro',
    'PREVOD Z UCTU': 'Transferencia Recebida',
    'VYBER': 'Saque em Dinheiro',
    'PREVOD NA UCET': 'Transferencia Enviada',
    'VYBER KARTOU': 'Saque com Cartao'
}

tipo_trans ={
    'POJISTNE': 'Pagamento de Seguro',
    'SLUZBY': 'Tarifa Bancaria',
    'UROK': 'Juros Recebidos',
    'SANKC. UROK': 'Juros de Multa',
    'SIPO': 'Contas de Consumo',
    'DUCHOD': 'Aposentadoria / Pensao',
    'UVER': 'Parcela de Emprestimo'
}




transacoes_df['tipo_movimentacao'] = transacoes_df['tipo_movimentacao'].replace(tipo_mov)
transacoes_df['modo_operacao'] = transacoes_df['modo_operacao'].replace(tipo_ope)
transacoes_df['categoria_transacao'] = transacoes_df['categoria_transacao'].replace(tipo_trans)
transacoes_df.sample(2)

In [ ]:
# Percentual de nulos — mais útil em bases grandes
# len(df) = número total de linhas
print('Percentual de nulos por coluna:')
print((transacoes_df.isnull().sum() / len(transacoes_df) * 100).round(2))

### Tratamento de Missing Values

In [ ]:
# Definindo valores substitutos que fazem sentido para o negócio do banco

# tabela transacoes_df

valores_substitutos = {
    'modo_operacao': 'Sistema / Automático',
    'categoria_transacao': 'Movimentação Geral',
    'banco_parceiro': 'Não se aplica (Interno)',
    'conta_parceiro': 'Não se aplica (Interno)'
}

# Aplicando nosso dicionario para preencher Missing Values
transacoes_df = transacoes_df.fillna(value=valores_substitutos)

In [ ]:
# Percentual de nulos — mais útil em bases grandes  (POS PROCESSAMENTO DOS NULOS)

print('Percentual de nulos por coluna:')
print((transacoes_df.isnull().sum() / len(transacoes_df) * 100).round(2))

In [ ]:
# Iremos preencher os Missing Values com modo de Imputaçao Estratificada preenchendo a taxa de desemprego com a mediana da própria região daquele distrito

# 1. Transformamos temporariamente para float para aceitar as medianas do groupby
distrito_df['crimes_cometidos_95'] = distrito_df['crimes_cometidos_95'].astype(float)

# 2. Agora aplicamos a imputação estratificada (vai rodar sem reclamar)
distrito_df['crimes_cometidos_95'] = distrito_df['crimes_cometidos_95'].fillna(
    distrito_df.groupby('regiao')['crimes_cometidos_95'].transform('median')
)

# 3. Como não existe "meio crime", limpamos a coluna transformando-a em inteiro puro
distrito_df['crimes_cometidos_95'] = distrito_df['crimes_cometidos_95'].astype(int)

# Aplicando a imputação estratificada na taxa de desemprego de 95
distrito_df['taxa_desemprego_95'] = distrito_df['taxa_desemprego_95'].fillna(
    distrito_df.groupby('regiao')['taxa_desemprego_95'].transform('median')
)


In [ ]:
distrito_df.sample(5)

### Tratamento de Outliers

In [ ]:
# Imputaçao Estratificada

transacoes_df_copy = transacoes_df


# Identificacao de Outliers com IQR

outliers = {}

# Filtragem por IQR

for col in transacoes_df_copy.select_dtypes(include=['float64', 'int64']).columns:
    Q1 = transacoes_df_copy[col].quantile(0.25)
    Q3 = transacoes_df_copy[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    filtro = (transacoes_df_copy[col] < limite_inferior) | (transacoes_df_copy[col] > limite_superior)
    count = filtro.sum()

    # Contagem de Outliers por coluna

    if count > 0:
        menor_valor = transacoes_df_copy.loc[filtro, col].min()
        maior_valor = transacoes_df_copy.loc[filtro, col].max()
    else:
        maior_valor = None
        menor_valor = None
    
    # Construindo um dataframe com os Outliers encontrados

    outliers[col] = {
        'Contagem de Outliers': count,
        '% de Outliers': ((count / len(transacoes_df_copy)) * 100).round(3),
        'Limite Inferior': limite_inferior.round(3),
        'Limite Superior': limite_superior.round(3),
        'Maior outlier': maior_valor,
        'Menor outlier': menor_valor
    }


outlier_df = pd.DataFrame.from_dict(outliers, orient='index')
outlier_df.head()

#### Motivo da Decisão

Durante a etapa de Análise Exploratória de Dados (EDA) na tabela de transações, identificamos uma alta volatilidade nos dados. Aplicamos o cálculo de Intervalo Interquartil (IQR) para mapear valores discrepantes e descobrimos que:

- Cerca de 11,38% da base da coluna valor_transacao (mais de 120 mil registros) está classificada matematicamente como outlier.

- Cerca de 3,29% da coluna saldo_apos_transacao (quase 35 mil registros) também se enquadram como outliers, incluindo saldos extremamente negativos.


Decidimos utilizar a Clusterização baseada em Regras de Negócio, aproveitando os limites matemáticos exatos que já havíamos calculado através do IQR.

Em vez de alterar ou excluir os dados, isolamos a base em dois DataFrames distintos, usando o Limite Superior de R$ 16.796,15 como linha de corte:

- Cluster Varejo Comum: Contém 88% das transações (valores até R$ 16.796). Representa o comportamento padrão do correntista pessoa física.

- Cluster Alta Renda / PJ: Contém os 11% de outliers (valores acima de R$ 16.796). Representa o comportamento de grandes movimentadores.

In [ ]:
# Utilização do limite superior feito pelo metodo de IQR
limite_sup_valor = 16796.150

# 2. Identificar as contas que possuem transações acima do limite (Outliers)
contas_alta_renda = transacoes_df_copy[transacoes_df_copy['valor_transacao'] > limite_sup_valor]['id_conta'].unique()

# 3. Descobrir quem são os donos dessas contas usando a tabela relacional 'cliente_conta_df'
# Filtramos por 'proprietario' para garantir que o verdadeiro dono receba o rótulo
clientes_alta_renda = cliente_conta_df[
    (cliente_conta_df['id_conta'].isin(contas_alta_renda)) & 
    (cliente_conta_df['tipo_acesso'] == 'proprietario')
]['id_cliente'].unique()

# 4. Enriquecer a Dimensão Cliente (clientes_df) criando o 'Atributo de Cluster'
import numpy as np

clientes_df['cluster_comportamental'] = np.where(
    clientes_df['id_cliente'].isin(clientes_alta_renda),
    'Alta Renda / PJ (Outlier)',
    'Varejo Comum (Padrão)'
)

# 5. Visualizar o resultado na tabela Dimensão
print("Distribuição dos Clientes por Cluster:")
print(clientes_df['cluster_comportamental'].value_counts())

display(clientes_df.sample(5))

In [ ]:
# Clusterização
limite_sup_valor = 16796.150



print(f"Tamanho do Varejo (Cluster A): {len(df_varejo)} transações")
print(f"Tamanho da Alta Renda (Cluster B): {len(df_alta_renda)} transações")

df_alta_renda

In [ ]:
# Indentificando a distribuiçao de Missing Values

(distrito_df.isnull().sum()*100/distrito_df.index.size).round(2).sort_values(ascending=False)

In [ ]:
#Calculando a Skewness

import scipy.stats as stats

g1 = stats.skew(transacoes_df_copy['valor_transacao'])

print(f'A skewness da valor_transacao é {g1}')

In [ ]:
#Função de análise gráfica

def comparar_variavel(dados, var_quantitativa, var_categorica):

    plt.figure(figsize=(14, 5))

    # Boxplot
    plt.subplot(1, 2, 1)
    sns.boxplot(data=dados, x=var_categorica, y=var_quantitativa)
    plt.title(f'Boxplot de {var_quantitativa} por {var_categorica}')
    plt.xticks()

    # KDE Plot
    plt.subplot(1, 2, 2)

    for categoria in dados[var_categorica].unique():
        subset = dados[dados[var_categorica] == categoria]
        sns.kdeplot(subset[var_quantitativa], label=categoria, fill=True, alpha=0.3)

    plt.title(f'Distribuição de {var_quantitativa} por {var_categorica}')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Recolhendo amostras
transacoes_df.sample(5)

In [ ]:
#Aplicação da função

comparar_variavel(df_varejo, var_quantitativa = 'saldo_apos_transacao', var_categorica='modo_operacao')